# Adaptive Supplier Risk Scoring System (ASRSS)

The ASRSS is a simulation designed as a proof of concept for dynamic supplier risk evaluation. It integrates data aggregation, contextual tagging, risk categorization, and predictive scoring to generate actionable insights. This system ensures adaptability by leveraging enriched data profiles and modular machine learning.

## Components:

1. **Synthetic Data Generator**: Simulates realistic supplier datasets, including financial, compliance, and real-time metrics, for system testing.
2. **Data Integration Engine (DIE)**: Aggregates, normalizes, and enriches data to create unified supplier profiles.
3. **Risk Categorization Engine (RCE)**: Dynamically tags suppliers with risk dimensions and categorizes risks using ML models.
4. **Predictive Scoring System (PSS)**: Calculates individual and aggregated risk scores for comprehensive supplier evaluation.

### Required Installations

Before running the Adaptive Supplier Risk Scoring System (ASRSS), ensure the necessary Python libraries are installed. Use the following commands in a code cell to install the dependencies:


In [ ]:
%pip install faker
%pip install fuzzywuzzy
%pip install python-Levenshtein

# Synthetic Data Generator

This program generates realistic, synthetic supplier data to simulate financial, compliance, and real-time metrics. It supports dynamic attributes like industry, region, and risk scores for testing data-driven systems. The generator also includes controlled duplication rates to evaluate entity resolution logic. This foundational dataset is used as input for the DIE to create enriched supplier profiles.

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
from tabulate import tabulate  # For pretty table formatting

# Initialize Faker for generating realistic synthetic data
fake = Faker()

# External constant for number of unique records
UNIQUE_RECORDS = 1000

# Function to generate synthetic supplier data
def generate_synthetic_data(unique_count=UNIQUE_RECORDS, dup_rate=0.1):
    """
    Generate synthetic supplier data, including financial, compliance, operational metrics, and questionnaire responses.
    """
    # Create unique supplier IDs to track individual suppliers
    supplier_ids = [fake.uuid4() for _ in range(unique_count)]

    # Define a pool of industries for diversity in supplier profiles
    industries = [
        'Manufacturing', 'Technology', 'Healthcare', 'Retail',
        'Energy', 'Finance', 'Transportation', 'Construction'
    ]

    # Financial data
    # Simulates metrics like revenue, debt, and credit scores
    # Example sources: Moody's (credit scores), Dun & Bradstreet (financial data)
    financial_data = pd.DataFrame({
        'SupplierID': supplier_ids,
        'SupplierName': [fake.company() for _ in range(unique_count)],  # Realistic company names
        'Revenue': np.random.randint(500000, 2000000, size=unique_count),  # Annual revenue, scaled to industry norms
        'Debt': np.random.randint(100000, 500000, size=unique_count),  # Debt, reflecting supplier liabilities
        'CreditScore': np.random.randint(300, 850, size=unique_count),  # Credit scores, indicative of financial health
        'Region': np.random.choice(['North America', 'Europe', 'Asia'], size=unique_count, p=[0.4, 0.3, 0.3]),  # Regional breakdown
        'Industry': np.random.choice(industries, size=unique_count, p=[0.15, 0.15, 0.1, 0.2, 0.1, 0.1, 0.1, 0.1])  # Industry tags
    })

    # Compliance data
    # Includes compliance scores, violations, and regulatory adherence
    # Example sources: LexisNexis (compliance records), Refinitiv World-Check (violations)
    compliance_data = pd.DataFrame({
        'SupplierID': supplier_ids,
        'SupplierName': [fake.company() for _ in range(unique_count)],
        'ComplianceScore': np.random.randint(50, 100, size=unique_count),  # Regulatory compliance ratings
        'Violations': np.random.randint(0, 10, size=unique_count),  # Number of violations recorded
        'RegulatoryAdherenceScore': np.random.uniform(0.5, 1.0, size=unique_count)  # Regulatory adherence metrics
    })

    # Operational data
    # Reflects operational risk via capacity, delivery performance, workforce turnover, and safety incidents
    # Example sources: Logistics performance reports, workforce audits
    operational_data = pd.DataFrame({
        'SupplierID': supplier_ids,
        'CapacityUtilization': np.random.uniform(0.6, 1.0, size=unique_count),  # Proportion of capacity utilized
        'OnTimeDeliveryRate': np.random.uniform(0.7, 1.0, size=unique_count),  # Timely delivery percentage
        'WorkforceTurnoverRate': np.random.uniform(0.05, 0.2, size=unique_count),  # Workforce turnover rates
        'SafetyIncidents': np.random.randint(0, 5, size=unique_count)  # Annual safety incident counts
    })

    # Questionnaire data
    # Represents responses from supplier self-assessments
    # Example sources: Supplier ESG surveys, certification reports
    questionnaire_data = pd.DataFrame({
        'SupplierID': supplier_ids,
        'ESGPractices': np.random.choice(['Excellent', 'Good', 'Average', 'Poor'], size=unique_count, p=[0.2, 0.4, 0.3, 0.1]),
        'CyberSecurityTraining': np.random.choice(['Yes', 'No'], size=unique_count, p=[0.8, 0.2]),
        'SupplyChainVisibility': np.random.choice(['High', 'Moderate', 'Low'], size=unique_count, p=[0.4, 0.4, 0.2]),
        'Certifications': np.random.choice(['ISO 9001', 'ISO 14001', 'None'], size=unique_count, p=[0.3, 0.3, 0.4])
    })

    # Real-time feed
    # Includes dynamic metrics like ESG scores, cybersecurity incidents, and geopolitical risks
    # Example sources: EcoVadis (ESG scores), Bitsight (cyber risk), global news aggregators
    real_time_feed = [
        {'SupplierID': supplier_ids[i],
         'SupplierName': financial_data.iloc[i]['SupplierName'],
         'ESGScore': np.random.randint(40, 100),  # Sustainability performance ratings
         'CyberSecurityIncident': np.random.choice([True, False], p=[0.1, 0.9]),  # Binary cybersecurity incident indicator
         'GeopoliticalRisk': np.random.uniform(0, 1)}  # Geopolitical risk scores
        for i in range(unique_count)
    ]

    # Introduce duplicates based on dup_rate
    # Helps simulate real-world scenarios where data redundancy exists
    num_dups = int(unique_count * dup_rate)
    for _ in range(num_dups):
        duplicate = financial_data.sample(1).iloc[0]
        financial_data = pd.concat([financial_data, pd.DataFrame([duplicate])], ignore_index=True)

    return financial_data, compliance_data, operational_data, questionnaire_data, real_time_feed

# Generate data
financial_data, compliance_data, operational_data, questionnaire_data, real_time_feed = generate_synthetic_data(unique_count=1000, dup_rate=0.1)

# Display samples with tabulate formatting for better readability
print("\nSample Financial Data:")
print(tabulate(financial_data.head(5), headers="keys", tablefmt="grid"))

print("\nSample Compliance Data:")
print(tabulate(compliance_data.head(5), headers="keys", tablefmt="grid"))

print("\nSample Operational Data:")
print(tabulate(operational_data.head(5), headers="keys", tablefmt="grid"))

print("\nSample Questionnaire Data:")
print(tabulate(questionnaire_data.head(5), headers="keys", tablefmt="grid"))

print("\nSample Real-Time Feed:")
print(tabulate(real_time_feed[:5], headers="keys", tablefmt="grid"))


# Data Integration Engine (DIE)

The DIE aggregates, normalizes, and resolves data from multiple sources into a unified format. It enriches supplier profiles with calculated fields like financial risk and tags for regions and industries. These profiles provide a consistent, clean dataset for downstream risk evaluation and scoring by the RCE and PSS.

In [ ]:
import pandas as pd
from fuzzywuzzy import fuzz
from tabulate import tabulate

# Step 1: Data Aggregation
def aggregate_data(financial_data, compliance_data, operational_data, questionnaire_data, real_time_feed):
    """
    Aggregates financial, compliance, operational, questionnaire, and real-time data into a unified format.
    Converts the real-time feed into a DataFrame for compatibility.

    Examples of data sources:
    - Financial data: Moody’s (credit scores), Dun & Bradstreet (financial metrics)
    - Compliance data: LexisNexis (regulatory adherence), Refinitiv World-Check (violations)
    - Operational data: Supplier performance reports, logistics providers
    - Questionnaire data: Supplier ESG surveys, certification audits
    - Real-time feeds: EcoVadis (ESG scores), Bitsight (cyber risk), news aggregators
    """
    print("\n--- Step 1: Aggregating Data ---")

    # Convert the real-time feed into a DataFrame
    real_time_df = pd.DataFrame(real_time_feed)

    # Merge all datasets on SupplierID for consistency
    aggregated_data = financial_data.merge(compliance_data, on='SupplierID', suffixes=('', '_comp'))
    aggregated_data = aggregated_data.merge(operational_data, on='SupplierID', suffixes=('', '_op'))
    aggregated_data = aggregated_data.merge(questionnaire_data, on='SupplierID', suffixes=('', '_ques'))
    aggregated_data = aggregated_data.merge(real_time_df, on='SupplierID', suffixes=('', '_rt'))

    print(f"Aggregated Dataset Shape: {aggregated_data.shape}")
    return aggregated_data

# Step 2: Validation and Normalization
def validate_and_normalize(data):
    """
    Validates data consistency and normalizes fields like SupplierName.
    Ensures numeric columns have no missing values by filling defaults.

    Examples of validations:
    - Normalize SupplierName for consistency (e.g., lowercase)
    - Fill missing numeric fields with zeros for calculations
    """
    print("\n--- Step 2: Validating and Normalizing Data ---")

    # Normalize SupplierName for consistency
    data['SupplierName'] = data['SupplierName'].str.lower()

    # Replace missing values in numeric fields with defaults
    numeric_columns = data.select_dtypes(include=['number']).columns
    data[numeric_columns] = data[numeric_columns].fillna(0)

    print(f"Validated and Normalized Dataset Shape: {data.shape}")
    return data

# Step 3: Entity Resolution
def resolve_entities(data):
    """
    Resolves duplicate entities using fuzzy matching on SupplierName to ensure unique records.
    Uses a similarity threshold to identify potential duplicates.

    Examples:
    - Resolve duplicates caused by inconsistent naming (e.g., "Acme Corp" vs "Acme Corporation")
    - Use fuzzy matching to compare SupplierName values
    """
    print("\n--- Step 3: Resolving Entities ---")

    unique_records = []
    resolved_ids = set()

    for _, row in data.iterrows():
        if row['SupplierID'] not in resolved_ids:
            similar = data[data['SupplierName'].apply(lambda x: fuzz.ratio(x, row['SupplierName']) > 90)]
            if similar.shape[0] > 1:
                resolved_ids.update(similar['SupplierID'])
                unique_records.append(row)
            else:
                unique_records.append(row)

    resolved_data = pd.DataFrame(unique_records)
    print(f"Records After Entity Resolution: {resolved_data.shape[0]}")
    return resolved_data

# Step 4: Data Transformation
def transform_to_profiles(data):
    """
    Transforms the integrated dataset into enriched supplier profiles with calculated metrics.

    Examples of transformations:
    - FinancialRisk: Calculate debt-to-revenue ratio
    - OperationalEfficiencyScore: Combine capacity utilization, delivery rate, and turnover rate
    - ESGAdherenceScore: Combine ESG practices, regulatory adherence, and ESG scores
    """
    print("\n--- Step 4: Transforming Data ---")

    # Financial risk: debt-to-revenue ratio
    data['FinancialRisk'] = data['Debt'] / data['Revenue'] * 100

    # Operational efficiency: weighted combination of metrics
    data['OperationalEfficiencyScore'] = (
        0.5 * data['CapacityUtilization'] +
        0.3 * data['OnTimeDeliveryRate'] -
        0.2 * data['WorkforceTurnoverRate']
    ).clip(0, 1)  # Clip to range [0, 1]

    # ESG adherence: composite score of regulatory adherence, ESG practices, and ESG score
    data['ESGAdherenceScore'] = (
        0.4 * data['RegulatoryAdherenceScore'] +
        0.3 * data['ESGScore'] +
        0.3 * data['ESGPractices'].map({'Excellent': 1.0, 'Good': 0.8, 'Average': 0.5, 'Poor': 0.2})
    ).fillna(0)

    print("Sample Enriched Profiles:")
    print(tabulate(data.head(5), headers="keys", tablefmt="grid"))
    return data

# Orchestrating the DIE
def data_integration_engine(financial_data, compliance_data, operational_data, questionnaire_data, real_time_feed):
    """
    Orchestrates the data integration process: aggregation, normalization, entity resolution, and transformation.
    """
    # Step 1: Aggregate data
    aggregated_data = aggregate_data(financial_data, compliance_data, operational_data, questionnaire_data, real_time_feed)

    # Step 2: Validate and normalize
    validated_data = validate_and_normalize(aggregated_data)

    # Step 3: Resolve duplicate entities
    resolved_data = resolve_entities(validated_data)

    # Step 4: Transform into enriched profiles
    enriched_profiles = transform_to_profiles(resolved_data)

    return enriched_profiles

# Execute the DIE with generated data
financial_data, compliance_data, operational_data, questionnaire_data, real_time_feed = generate_synthetic_data()
enriched_profiles = data_integration_engine(financial_data, compliance_data, operational_data, questionnaire_data, real_time_feed)

## Visualize Enriched Supplier Profiles

In [ ]:
# Save the enriched profiles to a single CSV file
enriched_profiles.to_csv('enriched_supplier_data.csv', index=False)
print("Enriched supplier profiles saved to enriched_supplier_data.csv.")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- Visualization 1: Supplier Distribution by Industry ---
def visualize_industry_distribution(data):
    plt.figure(figsize=(12, 8))
    industry_counts = data['Industry'].value_counts()
    industry_counts.plot(kind='bar', color='skyblue', edgecolor='black')
    plt.title('Supplier Distribution by Industry', fontsize=24)
    plt.xlabel('Industry', fontsize=18)
    plt.ylabel('Number of Suppliers', fontsize=18)
    plt.xticks(rotation=45, fontsize=14)
    plt.yticks(fontsize=14)
    plt.tight_layout()
    plt.show()

# --- Visualization 2: Box Plot of Revenue, Debt, and Compliance Score ---
def visualize_metric_distributions(data):
    metrics = ['Revenue', 'Debt', 'ComplianceScore']
    plt.figure(figsize=(12, 8))
    sns.boxplot(data=data[metrics])
    plt.title('Distribution of Key Metrics', fontsize=24)
    plt.xlabel('Metrics', fontsize=18)
    plt.ylabel('Values', fontsize=18)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.tight_layout()
    plt.show()

# --- Visualization 3: Pie Chart of ESG Practices ---
def visualize_esg_practices(data):
    esg_counts = data['ESGPractices'].value_counts()
    plt.figure(figsize=(10, 10))
    esg_counts.plot(kind='pie', autopct='%1.1f%%', colors=['#4CAF50', '#FF9800', '#03A9F4', '#E91E63'], startangle=140)
    plt.title('Distribution of ESG Practices', fontsize=24)
    plt.ylabel('')  # Hide y-axis label for aesthetic
    plt.tight_layout()
    plt.show()

# --- Visualization 4: Pair Plot of Quantitative Metrics ---
def visualize_pairwise_relationships(data):
    pair_data = data[['Revenue', 'Debt', 'CreditScore', 'ComplianceScore']].copy()
    sns.pairplot(pair_data, diag_kind='kde', corner=True, plot_kws={'alpha': 0.7})
    plt.suptitle('Pairwise Relationships Among Key Metrics', y=1.02, fontsize=24)
    plt.tight_layout()
    plt.show()

# --- Visualization 5: Supplier Risk Heatmap ---
def visualize_supplier_risk_heatmap(data):
    # Normalize the risk metrics for visualization
    data['FinancialRisk'] = (data['FinancialRisk'] - data['FinancialRisk'].min()) / (data['FinancialRisk'].max() - data['FinancialRisk'].min())
    data['OperationalEfficiencyScore'] = (data['OperationalEfficiencyScore'] - data['OperationalEfficiencyScore'].min()) / (data['OperationalEfficiencyScore'].max() - data['OperationalEfficiencyScore'].min())
    data['ESGAdherenceScore'] = (data['ESGAdherenceScore'] - data['ESGAdherenceScore'].min()) / (data['ESGAdherenceScore'].max() - data['ESGAdherenceScore'].min())

    # Select a subset of suppliers (e.g., top 50) for readability
    heatmap_data = data[['SupplierID', 'FinancialRisk', 'OperationalEfficiencyScore', 'ESGAdherenceScore']].copy()
    heatmap_data.set_index('SupplierID', inplace=True)
    heatmap_data = heatmap_data.head(50)  # Show only top 50 suppliers for clarity

    # Generate the heatmap
    plt.figure(figsize=(14, 10))
    sns.heatmap(heatmap_data, cmap='coolwarm', annot=False, cbar=True, linewidths=0.5)
    plt.title('Supplier Risk Heatmap (Normalized)', fontsize=24)
    plt.xlabel('Risk Metrics', fontsize=18)
    plt.ylabel('Supplier ID', fontsize=18)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=12)
    plt.tight_layout()
    plt.show()

# --- Main Execution ---
if __name__ == "__main__":

    # Create a deep copy of enriched_profiles
    profiles = enriched_profiles.copy(deep=True)

    # Visualization 1: Industry distribution
    visualize_industry_distribution(profiles)

    # Visualization 2: Metric distributions
    visualize_metric_distributions(profiles)

    # Visualization 3: ESG practices
    visualize_esg_practices(profiles)

    # Visualization 4: Pairwise relationships
    visualize_pairwise_relationships(profiles)

    # Visualization 5: Supplier risk heatmap
    visualize_supplier_risk_heatmap(profiles)


# Risk Categorization Engine (RCE)

The RCE dynamically assigns risk dimensions (e.g., compliance, financial, geopolitical) to suppliers using contextual data like industry, region, and specific metrics. It employs modular ML models to categorize risks and detect anomalies, preparing enriched profiles with tagged risk dimensions for scoring in the PSS.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from tabulate import tabulate

# Step 1: Feature Extraction
# Extracts relevant features from the enriched dataset for processing by ML models.
def extract_features(data):
    """
    Extracts key features from the enriched profiles for ML categorization.
    Ensures data consistency and readiness for downstream processing.
    """
    features = data[['SupplierID', 'SupplierName', 'FinancialRisk', 'OperationalEfficiencyScore',
                     'ESGAdherenceScore', 'GeopoliticalRisk', 'SafetyIncidents']].copy()

    # Fill missing values with defaults to ensure no NaN issues
    features.fillna({
        'FinancialRisk': 0,
        'OperationalEfficiencyScore': 0.5,
        'ESGAdherenceScore': 0.5,
        'GeopoliticalRisk': 0,
        'SafetyIncidents': 0
    }, inplace=True)
    return features

# Step 2.1: Financial Risk Categorization
# Uses Logistic Regression to classify suppliers into financial risk categories.
def categorize_financial_risk(data):
    """
    Categorizes suppliers into financial risk levels: Low, Moderate, or High.
    Uses logistic regression for classification based on predefined bins.
    """
    lr = LogisticRegression(random_state=42)
    financial_features = data[['FinancialRisk']]
    financial_labels = pd.cut(
        data['FinancialRisk'],
        bins=[0, 50, 75, float('inf')],
        labels=['Low', 'Moderate', 'High'],
        right=False
    )
    lr.fit(financial_features, financial_labels)
    data['FinancialRiskCategory'] = lr.predict(financial_features)
    return data

# Step 2.2: ESG Risk Categorization
# Uses Decision Tree to classify suppliers into ESG risk levels.
def categorize_esg_risk(data):
    """
    Categorizes suppliers into ESG risk levels: Low, Moderate, or High.
    Utilizes a decision tree classifier with ESG adherence scores.
    """
    dt = DecisionTreeClassifier(random_state=42)
    esg_features = data[['ESGAdherenceScore']]
    esg_labels = pd.cut(
        data['ESGAdherenceScore'],
        bins=[0, 20, 50, float('inf')],
        labels=['Low', 'Moderate', 'High'],
        right=False
    )
    dt.fit(esg_features, esg_labels)
    data['ESGRiskCategory'] = dt.predict(esg_features)
    return data

# Step 2.3: Geopolitical Risk Clustering
# Uses KMeans clustering to group suppliers based on geopolitical risk levels.
def cluster_geopolitical_risk(data):
    """
    Clusters suppliers into geopolitical risk groups using KMeans clustering.
    This provides insight into regional instability or related factors.
    """
    scaler = StandardScaler()
    geopolitical_scaled = scaler.fit_transform(data[['GeopoliticalRisk']])
    kmeans = KMeans(n_clusters=3, random_state=42)
    data['GeopoliticalRiskCluster'] = kmeans.fit_predict(geopolitical_scaled)
    return data

# Step 3: Anomaly Detection
# Uses Isolation Forest to identify anomalies in supplier profiles.
def detect_anomalies(data):
    """
    Identifies anomalies in supplier profiles using Isolation Forest.
    Flags suppliers with unusual patterns in numeric features.
    """
    numeric_features = data[['FinancialRisk', 'OperationalEfficiencyScore', 'ESGAdherenceScore',
                             'GeopoliticalRisk', 'SafetyIncidents']]
    isolation_forest = IsolationForest(contamination=0.05, random_state=42)
    data['Anomaly'] = isolation_forest.fit_predict(numeric_features)
    data['Anomaly'] = data['Anomaly'].map({1: 'Normal', -1: 'Anomaly'})
    return data

# Step 4: Enhance Profiles with Risk Dimensions and Explanations
# Adds risk dimensions, anomaly explanations, and weighted scores to profiles.
def enhance_profiles(data):
    """
    Adds dynamic risk dimensions, anomaly explanations, and weighted risk scores.
    Illustrates the modular and adaptive capabilities of the RCE.
    """
    data['RiskDimensions'] = data.apply(
        lambda row: [dim for dim, cond in [
            ('Financial', row['FinancialRiskCategory'] != 'Low'),
            ('ESG', row['ESGRiskCategory'] != 'Low'),
            ('Geopolitical', row['GeopoliticalRiskCluster'] > 0),
            ('Safety', row['SafetyIncidents'] > 1)
        ] if cond], axis=1)

    data['AnomalyExplanation'] = data.apply(
        lambda row: "High Safety Incidents" if row['SafetyIncidents'] > 2 else
                    "Unusual Financial Stability" if row['Anomaly'] == 'Anomaly' and row['FinancialRisk'] > 50 else
                    "None", axis=1)

    data['WeightedRiskScore'] = data.apply(
        lambda row: (
            (row['FinancialRisk'] * 0.4) +
            (row['OperationalEfficiencyScore'] * 0.2) +
            (row['ESGAdherenceScore'] * 0.3) +
            (row['GeopoliticalRisk'] * 0.1)
        ), axis=1)

    return data

# Orchestrating the RCE
# Executes all steps of the RCE: feature extraction, categorization, anomaly detection, and enhancement.
def risk_categorization_engine(enriched_profiles):
    """
    Executes the Risk Categorization Engine (RCE) pipeline:
    1. Extract features from enriched profiles.
    2. Categorize financial, ESG, and geopolitical risks.
    3. Detect anomalies in supplier profiles.
    4. Enhance profiles with risk dimensions and weighted scores.
    """
    profiles = extract_features(enriched_profiles)
    profiles = categorize_financial_risk(profiles)
    profiles = categorize_esg_risk(profiles)
    profiles = cluster_geopolitical_risk(profiles)
    profiles = detect_anomalies(profiles)
    profiles = enhance_profiles(profiles)
    return profiles

# Execute the RCE with enriched profiles
categorized_profiles = risk_categorization_engine(enriched_profiles)

# Display only the first n rows of the final enhanced profiles
print("\n--- Final Categorized Profiles (First n Rows) ---")
print(tabulate(categorized_profiles[['SupplierName', 'RiskDimensions', 'Anomaly', 'AnomalyExplanation', 'WeightedRiskScore']].head(12),
               headers="keys", tablefmt="grid"))



In [ ]:
# Save the enriched profiles to a single CSV file
categorized_profiles.to_csv('categorized_supplier_data.csv', index=False)
print("Enriched & Categorized supplier profiles saved to categorized_supplier_data.csv.")

# Predictive Scoring System (PSS)

The PSS generates dynamic risk scores for each supplier by evaluating tagged dimensions from the RCE. It assigns scores for compliance, financial, geopolitical, and cybersecurity risks, then aggregates them into an overall risk score. This system enables predictive and context-aware supplier risk evaluation.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LogisticRegression
from tabulate import tabulate

# PREDICTIVE SCORING SYSTEM (PSS)
# This program calculates comprehensive Supplier Risk Scores by analyzing enriched profiles.
# Risk scores are dynamically weighted based on tagged Risk Dimensions.

# Step 1: Predict Financial Risk Scores
def predict_financial_risk(data):
    """
    Predict financial risk scores using a Gradient Boosting Regressor (GBR).
    """
    print("--- Step 1.1: Predicting Financial Risk Scores ---")
    financial_features = data[['FinancialRisk']]
    gbr = GradientBoostingRegressor(random_state=42)
    gbr.fit(financial_features, data['FinancialRisk'])
    data['FinancialRiskScore'] = gbr.predict(financial_features)
    return data

# Step 2: Predict ESG Risk Scores
def predict_esg_risk(data):
    """
    Predict ESG risk scores using a Support Vector Regressor (SVR).
    """
    print("--- Step 1.2: Predicting ESG Risk Scores ---")
    esg_features = data[['ESGAdherenceScore']]
    svr = SVR()
    svr.fit(esg_features, data['ESGAdherenceScore'])
    data['ESGRiskScore'] = svr.predict(esg_features)
    return data

# Step 3: Predict Operational Risk Scores
def predict_operational_risk(data):
    """
    Predict operational risk scores using a Logistic Regression model.
    """
    print("--- Step 1.3: Predicting Operational Risk Scores ---")
    operational_features = data[['OperationalEfficiencyScore']]
    operational_labels = pd.cut(data['OperationalEfficiencyScore'], bins=[0, 0.6, 0.7, 1.0], labels=[80, 50, 20])
    lr = LogisticRegression(random_state=42)
    lr.fit(operational_features, operational_labels.to_numpy().ravel())
    data['OperationalRiskScore'] = lr.predict_proba(operational_features)[:, 1] * 100
    return data

# Step 4: Predict Geopolitical Risk Scores
def predict_geopolitical_risk(data):
    """
    Map GeopoliticalRisk directly as the risk score for simplicity.
    """
    print("--- Step 1.4: Predicting Geopolitical Risk Scores ---")
    data['GeopoliticalRiskScore'] = data['GeopoliticalRisk'] * 100
    return data

# Step 5: Aggregate Risk Scores
def aggregate_risk_scores(data):
    """
    Aggregates individual risk scores into an overall supplier risk score using dynamic weights.
    """
    print("--- Step 2: Aggregating Risk Scores ---")
    def calculate_weights(row):
        dimensions = row['RiskDimensions']
        base_weights = {'Financial': 0.4, 'ESG': 0.3, 'Operational': 0.2, 'Geopolitical': 0.1}
        dynamic_weights = {key: (base_weights[key] if key in dimensions else 0) for key in base_weights}
        total_weight = sum(dynamic_weights.values())
        if total_weight > 0:
            normalized_weights = {k: v / total_weight for k, v in dynamic_weights.items()}
        else:
            normalized_weights = base_weights  # Default to base weights if no dimensions are tagged
        return normalized_weights

    weights = data.apply(calculate_weights, axis=1)
    data['OverallRiskScore'] = data.apply(
        lambda row: sum(
            [
                row['FinancialRiskScore'] * weights[row.name]['Financial'],
                row['ESGRiskScore'] * weights[row.name]['ESG'],
                row['OperationalRiskScore'] * weights[row.name]['Operational'],
                row['GeopoliticalRiskScore'] * weights[row.name]['Geopolitical']
            ]
        ),
        axis=1
    )
    return data

# Orchestrating the PSS
def predictive_scoring_system(data):
    """
    Orchestrates the predictive scoring system by applying modular ML-based risk scoring.
    """
    data = predict_financial_risk(data)
    data = predict_esg_risk(data)
    data = predict_operational_risk(data)
    data = predict_geopolitical_risk(data)
    data = aggregate_risk_scores(data)
    return data

# Example data structure from RCE (sample enriched profiles)
data = pd.DataFrame({
    'SupplierName': ['lopez-espinoza', 'anderson ltd', 'harrell-heath', 'pace-hale', 'knox, fuentes and miller'],
    'FinancialRisk': [11.4, 39.9, 29.7, 23.9, 33.9],
    'ESGAdherenceScore': [17.0, 22.0, 22.5, 27.2, 28.1],
    'OperationalEfficiencyScore': [0.65, 0.58, 0.59, 0.62, 0.54],
    'GeopoliticalRisk': [0.64, 0.73, 0.83, 0.79, 0.01],
    'RiskDimensions': [['Financial'], ['Financial', 'ESG'], ['Financial', 'ESG', 'Operational'], ['Financial', 'Geopolitical'], []]
})

# Execute the PSS with categorized profiles
final_profiles = predictive_scoring_system(data)

# Display the final scored profiles
print("\n--- Final Scored Profiles (First 5 Rows) ---")
print(tabulate(final_profiles[['SupplierName', 'FinancialRiskScore', 'ESGRiskScore', 'OperationalRiskScore', 'GeopoliticalRiskScore', 'OverallRiskScore']],
               headers="keys", tablefmt="grid"))


In [ ]:
# Save the final profiles to a single CSV file
final_profiles.to_csv('scored_supplier_data.csv', index=False)
print("Enriched & Categorized with Risk Scores, supplier profiles saved to scored_supplier_data.csv.")

## Visualize Category and Risk Scores

In [ ]:
# --- Normalization Function ---
def normalize_column(data, column_name):
    """
    Normalizes a column in the DataFrame to the range [0, 1].
    If all values in the column are the same, it sets them to 0.5.
    """
    min_val = data[column_name].min()
    max_val = data[column_name].max()

    if min_val == max_val:
        print(f"Warning: All values in {column_name} are the same.")
        data[column_name] = 0.5
    else:
        data[column_name] = (data[column_name] - min_val) / (max_val - min_val)

    return data

# --- Visualization 3: Supplier Risk Dashboard ---
def visualize_supplier_dashboard(data, supplier_index=0):
    """
    Displays a dashboard-like visualization for a single supplier,
    showing overall and individual risk scores.

    Parameters:
    - data: DataFrame containing supplier profiles with risk scores.
    - supplier_index: Index of the supplier to display (default is 0 for the first supplier).
    """
    # Normalize all risk columns before plotting
    for column in ['FinancialRisk', 'OperationalEfficiencyScore', 'ESGAdherenceScore', 'GeopoliticalRisk', 'OverallRiskScore']:
        if column in data.columns:
            data = normalize_column(data, column)

    # Ensure the index is within bounds
    if supplier_index < 0 or supplier_index >= len(data):
        print(f"Error: Supplier index {supplier_index} is out of bounds.")
        return

    # Select the supplier row by index
    supplier_data = data.iloc[supplier_index]

    # Extract relevant scores
    scores = {
        'Financial Risk': supplier_data['FinancialRisk'],
        'Operational Risk': supplier_data['OperationalEfficiencyScore'],
        'ESG Risk': supplier_data['ESGAdherenceScore'],
        'Geopolitical Risk': supplier_data['GeopoliticalRisk'],
        'Overall Risk': supplier_data.get('OverallRiskScore', 0)  # Default 0 if missing
    }

    # Plot the dashboard
    plt.figure(figsize=(12, 7))
    categories = list(scores.keys())
    values = list(scores.values())
    bar_colors = ['#4CAF50', '#FF9800', '#03A9F4', '#E91E63', '#9C27B0']

    bars = plt.bar(categories, values, color=bar_colors, edgecolor='black', alpha=0.8)
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f"{yval:.2f}", ha='center', fontsize=12)

    # Set title and labels
    supplier_name = supplier_data['SupplierName']
    plt.title(f"Risk Dashboard: {supplier_name}", fontsize=24)
    plt.ylabel("Risk Score (0 - 1)", fontsize=18)
    plt.ylim(0, 1.2)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.tight_layout()
    plt.show()

# --- Main Execution ---
if __name__ == "__main__":
    # Normalize risk columns
    for col in ['FinancialRisk', 'OperationalEfficiencyScore', 'ESGAdherenceScore', 'GeopoliticalRisk', 'OverallRiskScore']:
        if col in final_profiles.columns:
            normalize_column(final_profiles, col)

    # Visualization 3: Supplier risk dashboards
    visualize_supplier_dashboard(final_profiles, supplier_index=0)  # First supplier
    visualize_supplier_dashboard(final_profiles, supplier_index=1)  # Second supplier
    visualize_supplier_dashboard(final_profiles, supplier_index=2)  # Twelfth supplier
